Imports

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, brier_score_loss

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

Loading

In [49]:
DATA_DIR = Path.cwd() / "data" / "raw"

AIRPORTS_3 = ["LAX", "ORD", "JFK"]
AIRPORTS_5 = ["LAX", "ORD", "JFK", "ATL", "DFW"]
AIRPORTS_8 = ["LAX", "ORD", "JFK", "ATL", "DFW", "SEA", "DEN", "MIA"]

In [50]:
def load_airports(airport_codes):
    dfs = []
    
    for airport in airport_codes:
        path = DATA_DIR / f"{airport}.csv"
        df = pd.read_csv(path)
        df["AirportFile"] = airport
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

Preprocessing

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "fall"


def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 22:
        return "evening"
    else:
        return "night"


def preprocess_flights(flights):
    flights = flights.copy()

    # Convert cancelled/delayed to clean true/false strings
    flights["Cancelled"] = flights["Cancelled"].astype(str).str.lower()
    flights["Delayed"] = flights["Delayed"].astype(str).str.lower()

    # Remove cancelled flights because cancellation is a different problem
    flights = flights[flights["Cancelled"] == "false"].copy()

    # Convert time
    flights["Time"] = pd.to_datetime(flights["Time"], errors="coerce")
    flights = flights.dropna(subset=["Time"])

    # Time features
    flights["Date"] = flights["Time"].dt.date
    flights["Month"] = flights["Time"].dt.month
    flights["DayOfWeek"] = flights["Time"].dt.dayofweek
    flights["DepHour"] = flights["Time"].dt.hour
    flights["Season"] = flights["Month"].apply(get_season)
    flights["TimeOfDay"] = flights["DepHour"].apply(get_time_of_day)

    # Airport congestion = flights from same origin on same date/hour
    flights["AirportCongestionCount"] = (
        flights
        .groupby(["Origin", "Date", "DepHour"])["Origin"]
        .transform("count")
    )

    # Rank avoids qcut errors from duplicate values
    flights["AirportCongestion"] = pd.qcut(
        flights["AirportCongestionCount"].rank(method="first"),
        q=3,
        labels=["low", "medium", "high"]
    )

    return flights



In [60]:
flights_clean = preprocess_flights(flights)
flights_clean.head()

,Time,Origin,Dest,Carrier,Cancelled,CancellationReason,Delayed,DepDelayMinutes,CarrierDelay,WeatherDelay,...,Ice_Accretion_3hr,AirportFile,Date,Month,DayOfWeek,DepHour,Season,TimeOfDay,AirportCongestionCount,AirportCongestion
0,2021-01-01 09:00:00,LAX,JFK,American Airlines Inc.,false,NaN,false,0.0,NaN,NaN,...,0.0,LAX,2021-01-01,1,4,9,winter,morning,25,low
1,2021-01-02 09:00:00,LAX,JFK,American Airlines Inc.,false,NaN,true,2.0,NaN,NaN,...,0.0,LAX,2021-01-02,1,5,9,winter,morning,28,low
2,2021-01-03 09:00:00,LAX,JFK,American Airlines Inc.,false,NaN,true,28.0,28.0,0.0,...,0.0,LAX,2021-01-03,1,6,9,winter,morning,68,high
3,2021-01-03 09:00:00,LAX,JFK,American Airlines Inc.,false,NaN,true,28.0,28.0,0.0,...,0.0,LAX,2021-01-03,1,6,9,winter,morning,68,high
4,2021-01-04 09:00:00,LAX,JFK,American Airlines Inc.,false,NaN,false,0.0,NaN,NaN,...,0.0,LAX,2021-01-04,1,0,9,winter,morning,90,high


In [58]:
model_cols = [
    "Season",
    "DayOfWeek",
    "TimeOfDay",
    "Origin",
    "Carrier",
    "AirportCongestion",
    "Delayed"
]

data = flights_clean[model_cols].dropna()

# Sample so pgmpy does not take forever
data = data.sample(n=50000, random_state=42).reset_index(drop=True)

# Convert everything to strings for pgmpy
for col in data.columns:
    data[col] = data[col].astype(str)

data.head()

,Season,DayOfWeek,TimeOfDay,Origin,Carrier,AirportCongestion,Delayed
0,spring,1,afternoon,LAX,SkyWest Airlines Inc.,low,true
1,spring,1,afternoon,LAX,Spirit Air Lines,low,false
2,summer,6,morning,LAX,United Air Lines Inc.,high,false
3,winter,3,morning,LAX,United Air Lines Inc.,low,false
4,fall,6,evening,JFK,Delta Air Lines Inc.,low,true


Train Bayesian Network

In [54]:
train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

model = DiscreteBayesianNetwork([
    ("Season", "Delayed"),
    ("DayOfWeek", "Delayed"),
    ("TimeOfDay", "AirportCongestion"),
    ("Origin", "AirportCongestion"),
    ("AirportCongestion", "Delayed"),
    ("Origin", "Delayed"),
    ("Carrier", "Delayed")
])

model.fit(train_data)

In [55]:
from pgmpy.inference import VariableElimination
from sklearn.metrics import accuracy_score, f1_score, brier_score_loss

infer = VariableElimination(model)

# Use 2000 rows first because inference can be slow
test_sample = test_data.sample(n=2000, random_state=42)

pred_probs = []
true_labels = []

for _, row in test_sample.iterrows():
    evidence = row.drop("Delayed").to_dict()

    result = infer.query(
        variables=["Delayed"],
        evidence=evidence,
        show_progress=False
    )

    delayed_states = result.state_names["Delayed"]
    true_index = delayed_states.index("true")
    prob_delay = result.values[true_index]

    pred_probs.append(prob_delay)
    true_labels.append(1 if row["Delayed"] == "true" else 0)

pred_labels = [1 if p >= 0.5 else 0 for p in pred_probs]

print("Number evaluated:", len(true_labels))
print("Accuracy:", accuracy_score(true_labels, pred_labels))
print("F1 score:", f1_score(true_labels, pred_labels))
print("Brier score:", brier_score_loss(true_labels, pred_probs))

Number evaluated: 2000
Accuracy: 0.624
F1 score: 0.3785123966942149
Brier score: 0.23827815421107634


In [56]:
baseline_preds = [0] * len(true_labels)

print("Baseline accuracy:", accuracy_score(true_labels, baseline_preds))
print("Baseline F1:", f1_score(true_labels, baseline_preds))

Baseline accuracy: 0.63
Baseline F1: 0.0
